# Identifying Deepfakes - V29 The Pure Anchored K-Means

## The Logical Breakthrough
Instead of letting K-Means poison the Mahalanobis math with its 12% error rate, we combine the best of both worlds:
1. **The Unsupervised Macro-Structure:** We still use K-Means to cluster the entire dataset into 2 massive regions.
2. **The Pure Anchor Verification:** Inside each cluster, we look exclusively at the 5% training subset. We identify the "Majority Label" for that cluster, and then isolate *only* the True images of that label. We calculate the Covariance and Mean ($\mu, \Sigma$) using ONLY that uncorrupted subset!
3. **The Distance Rescue:** We measure every test image's distance to that perfectly pure anchor. If it strays too far, we flip it.

In [ ]:
import os, warnings, zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
import numpy as np
from scipy.spatial.distance import mahalanobis, cosine
from scipy.linalg import pinv
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = '/content'; FOLDER_PATH = '/content/drive/MyDrive/DataMining/project_dataset'
else:
    BASE_PATH = './'; FOLDER_PATH = './project_dataset'

REAL_IMAGE_DIR, FAKE_IMAGE_DIR = os.path.join(BASE_PATH, 'Real-img'), os.path.join(BASE_PATH, 'Image')
RESOLUTION, BATCH_SIZE, SEED = 224, 64, 2026

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def extract_if_needed(zip_name, target_dir):
    if not os.path.exists(target_dir) and os.path.exists(os.path.join(FOLDER_PATH, zip_name)):
        with zipfile.ZipFile(os.path.join(FOLDER_PATH, zip_name), 'r') as z: z.extractall(BASE_PATH)

extract_if_needed('Real-img.zip', REAL_IMAGE_DIR)
extract_if_needed('Fake-img.zip', FAKE_IMAGE_DIR)

class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.r = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(real_dir) else []
        self.f = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg', '.png'))] if os.path.exists(fake_dir) else []
        self.all_files = self.r + self.f
        self.labels = [0]*len(self.r) + [1]*len(self.f)
        self.transform = transform
    def __len__(self): return len(self.all_files)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.all_files[idx]).convert('RGB')
            if self.transform: img = self.transform(img)
            return img, self.labels[idx], self.all_files[idx]
        except: return torch.zeros((3, RESOLUTION, RESOLUTION)), self.labels[idx], self.all_files[idx]

full_dataset = DeepfakeDataset(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
]))
all_idx = list(range(len(full_dataset)))
np.random.shuffle(all_idx)

NETWORK_POOL_SIZE = int(len(full_dataset) * 0.05)  
network_loader = DataLoader(Subset(full_dataset, all_idx[:NETWORK_POOL_SIZE]), batch_size=BATCH_SIZE, shuffle=False)
ml_loader = DataLoader(Subset(full_dataset, all_idx[NETWORK_POOL_SIZE:]), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
locked_layers = ['conv1', 'bn1', 'layer1', 'layer2', 'layer3', 'layer4']
for name, param in model.named_parameters():
    if any(layer_name in name for layer_name in locked_layers): param.requires_grad = False
model.fc = nn.Linear(512, 2)
model = model.to(device)

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.CrossEntropyLoss()

if len(network_loader) > 0:
    for epoch in range(1):
        model.train()
        pbar = tqdm(network_loader, desc=f"Epoch 1/1")
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward(); optimizer.step()

In [ ]:
fc_layer = model.fc
model.fc = nn.Identity()
model.eval()

# We extract everything sequentially so we know exactly where the 5% pool ends
full_loader = DataLoader(Subset(full_dataset, all_idx), batch_size=BATCH_SIZE, shuffle=False)

latent_embeddings, weak_predictions, ground_truth = [], [], []

with torch.no_grad():
    for imgs, labels, _ in tqdm(full_loader, desc="Extraction"):
        imgs = imgs.to(device)
        latent = model(imgs)
        outputs = fc_layer(latent)
        _, preds = outputs.max(1)
        latent_embeddings.append(latent.cpu().numpy())
        weak_predictions.extend(preds.cpu().numpy())
        ground_truth.extend(labels.numpy())

latent_embeddings = np.vstack(latent_embeddings)
weak_predictions = np.array(weak_predictions)
ground_truth = np.array(ground_truth)

# Split into 5% Train (Anchor Base) and 95% Test (Mining Pool)
X_train = latent_embeddings[:NETWORK_POOL_SIZE]
y_train_true = ground_truth[:NETWORK_POOL_SIZE]
y_train_weak = weak_predictions[:NETWORK_POOL_SIZE]

X_test = latent_embeddings[NETWORK_POOL_SIZE:]
y_test_true = ground_truth[NETWORK_POOL_SIZE:]
y_test_weak = weak_predictions[NETWORK_POOL_SIZE:]

print(f"\n--> Raw ResNet Prediction Accuracy on Test Pool: {accuracy_score(y_test_true, y_test_weak):.4f}\n")

In [ ]:
print("Phase 1: Establishing Corrupted K-Means Baseline")
clusterer = KMeans(n_clusters=2, random_state=SEED, n_init=10)
clusterer.fit(latent_embeddings)

cluster_train = clusterer.labels_[:NETWORK_POOL_SIZE]
cluster_test = clusterer.labels_[NETWORK_POOL_SIZE:]

# We use the 5% labeled data to figure out which cluster is which
mean_0 = np.mean(y_train_true[cluster_train == 0])
mean_1 = np.mean(y_train_true[cluster_train == 1])

fake_cluster_idx = 0 if mean_0 > mean_1 else 1

test_baseline_preds = np.zeros(len(X_test))
test_baseline_preds[cluster_test == fake_cluster_idx] = 1

acc_base = accuracy_score(y_test_true, test_baseline_preds)
print(f"--> K-Means Unsupervised Baseline Accuracy on Test Pool: {acc_base:.4f}\n")

In [ ]:
print("Phase 2: The Pure Anchor Rescue Grid Search\n")

# Dictionaries to store distances and labels
cluster_data = {0: {'mah_dists': [], 'cos_dists': [], 'majority_label': 0},
                1: {'mah_dists': [], 'cos_dists': [], 'majority_label': 0}}

global_probs_mah = np.zeros(len(X_test))
global_probs_cos = np.zeros(len(X_test))

for i in range(2):
    test_mask = (cluster_test == i)
    train_mask = (cluster_train == i)
    
    majority_label = 1 if i == fake_cluster_idx else 0
    cluster_data[i]['majority_label'] = majority_label
    
    pure_mask = train_mask & (y_train_true == majority_label)
    X_pure = X_train[pure_mask]
    
    print(f"--- Cluster {i} ({'Fake' if majority_label==1 else 'Real'}) | Extracted Pure Anchor Size: {len(X_pure)} images ---")
    
    if len(X_pure) < 5: 
        print("Not enough pure samples for a stable anchor. Skipping rescue for this cluster.")
        continue

    mean_pure = np.mean(X_pure, axis=0)
    cov_pure = np.cov(X_pure, rowvar=False)
    pinv_cov = pinv(cov_pure)
    
    mah_dists = []
    cos_dists = []
    
    for x in X_test[test_mask]:
        mah_dists.append(mahalanobis(x, mean_pure, pinv_cov))
        cos_dists.append(cosine(x, mean_pure))
        
    mah_dists = np.array(mah_dists)
    cos_dists = np.array(cos_dists)
    
    cluster_data[i]['mah_dists'] = mah_dists
    cluster_data[i]['cos_dists'] = cos_dists
    
    global_probs_mah[test_mask] = mah_dists if majority_label == 1 else 1.0 - (mah_dists - mah_dists.min())/(mah_dists.max() - mah_dists.min() + 1e-9)
    global_probs_cos[test_mask] = cos_dists if majority_label == 1 else 1.0 - (cos_dists - cos_dists.min())/(cos_dists.max() - cos_dists.min() + 1e-9)

deviations_to_test = [1.0, 1.5, 2.0, 2.5, 3.0]

for sd_multiplier in deviations_to_test:
    grid_preds_mah = np.copy(test_baseline_preds)
    grid_preds_cos = np.copy(test_baseline_preds)
    
    for i in range(2):
        if len(cluster_data[i]['mah_dists']) == 0: continue
        
        test_mask = (cluster_test == i)
        majority_label = cluster_data[i]['majority_label']
        
        mah_dists = cluster_data[i]['mah_dists']
        mah_thresh = np.mean(mah_dists) + (sd_multiplier * np.std(mah_dists))
        mah_flips = mah_dists > mah_thresh
        grid_preds_mah[test_mask] = np.where(mah_flips, 1 - majority_label, majority_label)
        
        cos_dists = cluster_data[i]['cos_dists']
        cos_thresh = np.mean(cos_dists) + (sd_multiplier * np.std(cos_dists))
        cos_flips = cos_dists > cos_thresh
        grid_preds_cos[test_mask] = np.where(cos_flips, 1 - majority_label, majority_label)
        
    print("="*40)
    print(f"GRID SEARCH: FLIP AT {sd_multiplier} STANDARD DEVIATIONS")
    print("="*40)
    
    print("MAHALANOBIS RESCUE:")
    print(f"Global Accuracy:   {accuracy_score(y_test_true, grid_preds_mah):.4f}")
    print(f"Global Precision:  {precision_score(y_test_true, grid_preds_mah, zero_division=0):.4f}")
    print(f"Global Recall:     {recall_score(y_test_true, grid_preds_mah):.4f}")
    print(f"Cumulative AUC:    {roc_auc_score(y_test_true, global_probs_mah):.4f}\n")
    
    print("COSINE RESCUE:")
    print(f"Global Accuracy:   {accuracy_score(y_test_true, grid_preds_cos):.4f}")
    print(f"Global Precision:  {precision_score(y_test_true, grid_preds_cos, zero_division=0):.4f}")
    print(f"Global Recall:     {recall_score(y_test_true, grid_preds_cos):.4f}")
    print(f"Cumulative AUC:    {roc_auc_score(y_test_true, global_probs_cos):.4f}\n")
